# 06 – File Handling & Exception Management

Topics covered:
- Reading and writing text files
- Context managers (`with` statement)
- Working with CSV and JSON files
- Exception handling (`try` / `except` / `else` / `finally`)
- Raising and creating custom exceptions
- Best practices

## 1. Reading and Writing Text Files

Always use the `with` statement — it guarantees the file is closed even if an exception occurs.

In [ ]:
# Write a file
with open("/tmp/sample.txt", "w") as f:
    f.write("Line 1: Hello, Python!\n")
    f.write("Line 2: File handling is easy.\n")
    f.write("Line 3: Always use 'with'.\n")

print("File written.")

In [ ]:
# Read entire file at once
with open("/tmp/sample.txt", "r") as f:
    content = f.read()
print(content)

In [ ]:
# Read line by line — memory-efficient for large files
with open("/tmp/sample.txt") as f:
    for i, line in enumerate(f, 1):
        print(f"{i}: {line}", end="")

In [ ]:
# Read into a list of lines
with open("/tmp/sample.txt") as f:
    lines = f.readlines()   # includes '\n'
print(lines)

# Strip newlines
stripped = [l.rstrip() for l in lines]
print(stripped)

In [ ]:
# File modes
# 'r'  — read (default)
# 'w'  — write (creates or overwrites)
# 'a'  — append (creates or appends)
# 'x'  — exclusive create (fails if file exists)
# 'rb' — read binary
# 'w+' — read + write

# Append to file
with open("/tmp/sample.txt", "a") as f:
    f.write("Line 4: Appended line.\n")

with open("/tmp/sample.txt") as f:
    print(f.read())

## 2. Working with CSV Files

In [ ]:
import csv

# Write CSV
students = [
    {"name": "Alice", "age": 22, "score": 92},
    {"name": "Bob",   "age": 24, "score": 88},
    {"name": "Carol", "age": 23, "score": 95},
]

with open("/tmp/students.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "age", "score"])
    writer.writeheader()
    writer.writerows(students)

print("CSV written")

In [ ]:
# Read CSV
with open("/tmp/students.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(dict(row))

## 3. Working with JSON Files

In [ ]:
import json

config = {
    "model": "random_forest",
    "n_estimators": 100,
    "max_depth": 5,
    "features": ["age", "income", "credit_score"]
}

# Save
with open("/tmp/config.json", "w") as f:
    json.dump(config, f, indent=2)

# Load
with open("/tmp/config.json") as f:
    loaded_config = json.load(f)

print(loaded_config)
print(loaded_config["features"])

## 4. Exception Handling

```
try:
    risky code
except SomeError as e:
    handle error
else:
    runs only if NO exception
finally:
    ALWAYS runs (cleanup)
```

In [ ]:
# Basic exception handling
def divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Cannot divide by zero!")
        return None
    else:
        print(f"{a} / {b} = {result}")
        return result
    finally:
        print("divide() finished")  # runs regardless

divide(10, 2)
print("---")
divide(10, 0)

In [ ]:
# Catching multiple exceptions
def safe_convert(value, target_type):
    try:
        return target_type(value)
    except (ValueError, TypeError) as e:
        print(f"Conversion failed: {e}")
        return None

print(safe_convert("42", int))     # 42
print(safe_convert("hello", int))  # error
print(safe_convert(None, float))   # error

In [ ]:
# Common built-in exceptions
exceptions = [
    (ValueError,       lambda: int("abc")),
    (TypeError,        lambda: "a" + 1),
    (IndexError,       lambda: [1,2,3][10]),
    (KeyError,         lambda: {}["x"]),
    (AttributeError,   lambda: None.upper()),
    (FileNotFoundError,lambda: open("missing.txt")),
    (ZeroDivisionError,lambda: 1/0),
    (OverflowError,    lambda: 10.0**10000),
]

for exc_type, trigger in exceptions:
    try:
        trigger()
    except exc_type as e:
        print(f"{exc_type.__name__}: {e}")

In [ ]:
# Exception hierarchy — catch specific before general
def read_file_safe(path):
    try:
        with open(path) as f:
            return f.read()
    except FileNotFoundError:
        print(f"File not found: {path}")
    except PermissionError:
        print(f"No permission to read: {path}")
    except OSError as e:
        print(f"OS error: {e}")
    return None

content = read_file_safe("/tmp/sample.txt")   # works
content = read_file_safe("/tmp/missing.txt")  # caught cleanly

## 5. Raising Exceptions

In [ ]:
def set_age(age):
    if not isinstance(age, int):
        raise TypeError(f"age must be int, got {type(age).__name__}")
    if age < 0 or age > 150:
        raise ValueError(f"age must be 0-150, got {age}")
    return age

try:
    set_age("thirty")
except TypeError as e:
    print(e)

try:
    set_age(200)
except ValueError as e:
    print(e)

## 6. Custom Exceptions

Create domain-specific exceptions by subclassing `Exception`.

In [ ]:
class ModelNotTrainedError(Exception):
    """Raised when prediction is attempted before training."""
    pass


class DataValidationError(ValueError):
    """Raised when input data fails validation."""
    def __init__(self, column, message):
        self.column = column
        super().__init__(f"Column '{column}': {message}")


class SimpleClassifier:
    def __init__(self):
        self._trained = False

    def predict(self, X):
        if not self._trained:
            raise ModelNotTrainedError("Call .fit() before .predict()")
        return [0] * len(X)


clf = SimpleClassifier()
try:
    clf.predict([1, 2, 3])
except ModelNotTrainedError as e:
    print(f"ModelNotTrainedError: {e}")

try:
    raise DataValidationError("age", "contains negative values")
except DataValidationError as e:
    print(f"DataValidationError: {e} (column={e.column})")

## 7. Context Managers

The `with` statement works with any object that implements `__enter__` and `__exit__`.  
Build your own with `contextlib`.

In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def timer(label=""):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"{label} took {elapsed:.4f}s")

with timer("list comprehension"):
    squares = [x**2 for x in range(1_000_000)]

## Best Practices

| Practice | Reason |
|----------|--------|
| Always use `with` for files | Guarantees file is closed, even on exception |
| Catch specific exceptions | `except Exception` hides bugs |
| Don't swallow exceptions silently | At minimum, log the error |
| Use `else` for success path | Keeps try block minimal |
| Use `finally` for cleanup | Runs regardless of success/failure |
| Raise early, catch late | Validate inputs at function entry |

**Next →** [07 – OOP](../07-oops/)